# AI-Powered Research Paper Discovery System

An NLP-based research discovery pipeline that retrieves papers using **title and abstract semantics**, combines **semantic and keyword-based ranking**, and enriches the retrieved results with **named entities, keyphrases, AI-generated summaries, and explainable relevance evidence**.

### Pipeline
`Research Query → SentenceTransformer → FAISS Semantic Retrieval → Hybrid Ranking → NER → KeyBERT → BART Summarization → Relevance Explanation`

### Core Components
- Sentence embeddings with `all-MiniLM-L6-v2`
- FAISS vector similarity search
- Keyword-aware hybrid ranking
- Named Entity Recognition (NER)
- Keyphrase extraction using KeyBERT
- BART abstractive summarization
- Explainable paper-query relevance

> **Note:** The notebook keeps the initial embedding and semantic-similarity experiments for learning and project understanding before the final integrated pipeline.


In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("CShorten/ML-ArXiv-Papers" , split='train')

In [ ]:
print(dataset)

In [ ]:
dataset[0]

In [ ]:
import pandas as pd

In [ ]:
df=pd.DataFrame(dataset)
df

In [ ]:
df.columns

In [ ]:
df=df[['title','abstract']]
df

In [ ]:
df.shape

In [ ]:
df=df.head(15000)

In [ ]:
df.shape

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
df["paper_text"]=df["title"]+" "+df["abstract"]

In [ ]:
df[["paper_text"]].head()

In [ ]:
type(df[["paper_text"]])

In [ ]:
df[["paper_text"]]

In [ ]:
print(df["paper_text"].iloc[0])

In [ ]:
df=df.head(15000)

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
print(type(model))

In [ ]:
sample_text=df["paper_text"].iloc[0]
sample_text


In [ ]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

In [ ]:
sample_text=df["paper_text"].iloc[0]
sample_text

In [ ]:
embedding=model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

In [ ]:
embedding[:56]

In [ ]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())
sample_embedding

In [ ]:
sample_embedding.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1, -1),sample_embedding[0].reshape(1, -1))
print(similarity)

In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1, -1),sample_embedding[1].reshape(1, -1))
print(similarity)

In [ ]:
for i in range(1,5):
  sim=cosine_similarity(sample_embedding[0].reshape(1, -1),sample_embedding[i].reshape(1, -1))
  print(sim)

# FINAL RESEARCH PAPER DISCOVERY PIPELINE

## 1. Install required libraries

In [ ]:
!pip install -q faiss-cpu keybert transformers sentence-transformers


## 2. Mount Google Drive and load saved embeddings + FAISS index

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import faiss

PROJECT_PATH = "/content/drive/MyDrive/ML_INTERN_P2"
EMBEDDING_PATH = os.path.join(PROJECT_PATH, "paper_embeddings.npy")
FAISS_PATH = os.path.join(PROJECT_PATH, "paper_faiss.index")

if not os.path.exists(EMBEDDING_PATH) or not os.path.exists(FAISS_PATH):
    raise FileNotFoundError(
        "Precomputed FAISS assets were not found. "
        "Create a Google Drive folder named 'ML_INTERN_P2' and add "
        "'paper_embeddings.npy' and 'paper_faiss.index'. "
        "These large precomputed assets are intentionally not stored in the GitHub repository."
    )

embeddings = np.load(EMBEDDING_PATH)
index = faiss.read_index(FAISS_PATH)

if len(df) != index.ntotal:
    raise ValueError(
        f"Dataset/index mismatch: dataframe has {len(df)} papers "
        f"but FAISS contains {index.ntotal} vectors."
    )

print("Saved embeddings and FAISS index loaded successfully!")
print("Embeddings shape:", embeddings.shape)
print("Total papers in FAISS:", index.ntotal)


## 3. Hybrid Search

In [ ]:
def keyword_score(query, text):
    query_words = set(query.lower().split())
    text_words = set(str(text).lower().split())

    if len(query_words) == 0:
        return 0.0

    matched_words = query_words.intersection(text_words)
    return len(matched_words) / len(query_words)


def hybrid_search(query, k=5, semantic_weight=0.8, keyword_weight=0.2):
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)

    search_k = min(max(k * 10, k), index.ntotal)
    semantic_scores, paper_indices = index.search(query_embedding, search_k)

    results = []

    for semantic_score, idx in zip(semantic_scores[0], paper_indices[0]):
        if idx < 0:
            continue

        text = df.iloc[idx]["paper_text"]
        kw_score = keyword_score(query, text)

        hybrid_score = (
            semantic_weight * float(semantic_score)
            + keyword_weight * float(kw_score)
        )

        results.append({
            "index": int(idx),
            "semantic_score": float(semantic_score),
            "keyword_score": float(kw_score),
            "hybrid_score": float(hybrid_score)
        })

    results = sorted(
        results,
        key=lambda item: item["hybrid_score"],
        reverse=True
    )

    return results[:k]


## 4. Named Entity Recognition

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

def extract_entities(text):
    text = str(text)[:2000]
    entities = ner_pipeline(text)

    clean_entities = []
    seen = set()

    unwanted = {
        "of improvement", "the", "and", "model", "method"
    }

    for entity in entities:
        word = entity["word"].strip()
        score = float(entity["score"])

        if "##" in word or len(word) <= 1 or score < 0.70:
            continue

        if word.lower() in unwanted or word.lower() in seen:
            continue

        clean_entities.append({
            "entity": word,
            "type": entity["entity_group"],
            "score": round(score, 3)
        })

        seen.add(word.lower())

    return clean_entities


## 5. Keyphrase Extraction with KeyBERT

In [ ]:
from keybert import KeyBERT

keyword_model = KeyBERT(model=model)

def extract_keywords(text, top_n=5):
    keywords = keyword_model.extract_keywords(
        str(text),
        keyphrase_ngram_range=(1, 3),
        stop_words="english",
        top_n=top_n
    )

    return [
        {
            "keyword": keyword,
            "score": round(float(score), 3)
        }
        for keyword, score in keywords
    ]


## 6. BART Summarization

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_tokenizer = AutoTokenizer.from_pretrained(
    "facebook/bart-large-cnn"
)

bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/bart-large-cnn"
)

def generate_summary(text, max_length=120, min_length=40):
    text = " ".join(str(text).split())

    inputs = bart_tokenizer(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    )

    summary_ids = bart_model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_beams=4,
        max_length=max_length,
        min_length=min_length,
        early_stopping=True
    )

    return bart_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )


## 7. Explainable Relevance

In [ ]:
def generate_relevance_explanation(query, abstract, result):
    stop_words = {
        "a", "an", "the", "in", "on", "of", "for",
        "to", "and", "or", "with", "using", "by"
    }

    query_words = {
        word for word in query.lower().split()
        if word not in stop_words
    }

    abstract_words = set(
        str(abstract)
        .lower()
        .replace(",", " ")
        .replace(".", " ")
        .split()
    )

    matched_words = sorted(query_words.intersection(abstract_words))
    paper_keywords = extract_keywords(abstract, top_n=3)
    keyword_names = [item["keyword"] for item in paper_keywords]

    semantic_score = result["semantic_score"]

    if semantic_score >= 0.70:
        semantic_text = "strong semantic similarity"
    elif semantic_score >= 0.55:
        semantic_text = "moderate semantic similarity"
    else:
        semantic_text = "some semantic similarity"

    explanation = (
        f"This paper shows {semantic_text} with the user query "
        f"(semantic score: {semantic_score:.3f}). "
    )

    if matched_words:
        explanation += (
            "It directly matches important query terms such as "
            f"{', '.join(matched_words)}. "
        )

    if keyword_names:
        explanation += (
            "The paper mainly discusses topics such as "
            f"{', '.join(keyword_names)}, which makes it relevant "
            "to the requested research topic."
        )

    return explanation


### Integrated Search Architecture

The final function retrieves the highest-ranked papers from the hybrid search layer and then performs post-retrieval NLP analysis. NER and KeyBERT provide interpretable paper features, BART produces concise summaries, and the relevance explanation reports semantic and lexical evidence behind each recommendation.


## 8. Final Research Paper Search Pipeline

In [ ]:
def research_paper_search(query, k=5):
    results = hybrid_search(query, k=k)

    print("QUERY:", query)
    print("=" * 100)

    for rank, result in enumerate(results, start=1):
        idx = result["index"]

        title = df.iloc[idx]["title"]
        abstract = df.iloc[idx]["abstract"]

        entities = extract_entities(abstract)
        keywords = extract_keywords(abstract, top_n=5)
        summary = generate_summary(abstract)
        relevance_explanation = generate_relevance_explanation(
            query, abstract, result
        )

        print(f"\nRANK {rank}")

        print("\nTITLE:")
        print(title)

        print("\nSEARCH SCORES:")
        print("Semantic Score:", round(result["semantic_score"], 3))
        print("Keyword Score:", round(result["keyword_score"], 3))
        print("Hybrid Score:", round(result["hybrid_score"], 3))

        print("\nWHY THIS PAPER MATCHES YOUR QUERY:")
        print(relevance_explanation)

        print("\nAI GENERATED SUMMARY:")
        print(summary)

        print("\nNAMED ENTITIES:")
        if not entities:
            print("No named entities detected")
        else:
            for entity in entities:
                print(
                    "-", entity["entity"],
                    "| Type:", entity["type"],
                    "| Score:", entity["score"]
                )

        print("\nKEYWORDS:")
        for item in keywords:
            print(
                "-", item["keyword"],
                "| Score:", item["score"]
            )

        print("\n" + "-" * 100)


## 9. Final Test

In [ ]:
research_paper_search(
    "Deep learning in medical imaging",
    k=3
)
